# Stage 6B: stable PF64 bimodal overlay

This is a new leakage-controlled candidate, not the public-kernel reproduction. It keeps the promoted Stage 4 prediction and uses a stable 4 x 16-seed PF only to detect ambiguous two-mode wells. Four wells are processed concurrently.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
data_dir = Path('/content/rogii-data')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'

if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
drive_data = drive_root / 'data'
assert (drive_data / 'train').is_dir(), f'Missing train directory: {drive_data / "train"}'
if not (data_dir / 'train').is_dir():
    shutil.copytree(drive_data, data_dir, dirs_exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
print('repository:', repo_dir)
print('artifacts:', artifact_dir)

## 8-well smoke test
This validates code and data wiring only. Its RMSE is not a promotion result.

In [ ]:
SMOKE_RUN_ID = 'stage6_pf64_mha_diagnostic_smoke_v001'
smoke_dir = artifact_dir / SMOKE_RUN_ID
if not (smoke_dir / 'metrics.json').is_file():
    assert not smoke_dir.exists() or not any(smoke_dir.iterdir()), f'Incomplete run exists: {smoke_dir}'
    subprocess.run([
        'uv', 'run', 'rogii-train',
        '--config', 'configs/experiment/stage6_pf64_stable_mha.yaml',
        '--run-id', SMOKE_RUN_ID,
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
        '--limit-wells', '8',
    ], cwd=repo_dir, check=True)
json.loads((smoke_dir / 'metrics.json').read_text())

## Full PF64 diagnostic
Four independent 16-seed likelihood ensembles are averaged to avoid the instability observed when 128 seeds compete in one winner pool. This run supplies guarded two-mode statistics; it does not replace Stage 4 predictions.

In [ ]:
BASE_RUN_ID = 'stage6_pf64_mha_diagnostic_full_v001'
base_dir = artifact_dir / BASE_RUN_ID
if not (base_dir / 'metrics.json').is_file():
    assert not base_dir.exists() or not any(base_dir.iterdir()), f'Incomplete run exists: {base_dir}'
    subprocess.run([
        'uv', 'run', 'rogii-train',
        '--config', 'configs/experiment/stage6_pf64_stable_mha.yaml',
        '--run-id', BASE_RUN_ID,
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
    ], cwd=repo_dir, check=True)
diagnostic_metrics = json.loads((base_dir / 'metrics.json').read_text())
{'diagnostic_only_rmse': diagnostic_metrics['pooled_rmse']}

## Ensure Stage 4 exists, then apply the guarded MHA overlay
Missing Stage 2/3/4 prerequisites are created automatically. Existing complete Drive artifacts are reused.

In [ ]:
STAGE2_RUN_ID = 'stage2_pf_trend_blend_full_v001'
stage2_dir = artifact_dir / STAGE2_RUN_ID
if not (stage2_dir / 'metrics.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-train', '--config', 'configs/experiment/pf_trend_blend.yaml',
        '--run-id', STAGE2_RUN_ID, '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir),
    ], cwd=repo_dir, check=True)

STAGE3_RUN_ID = 'stage3_residual_hgb_full_v001'
stage3_dir = artifact_dir / STAGE3_RUN_ID
if not (stage3_dir / 'metrics.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-residual', '--config', 'configs/experiment/stage3_residual_hgb.yaml',
        '--base-run', str(stage2_dir), '--run-id', STAGE3_RUN_ID,
        '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir),
    ], cwd=repo_dir, check=True)

STAGE4_RUN_ID = 'stage4_tail_path_full_v001'
stage4_dir = artifact_dir / STAGE4_RUN_ID
if not (stage4_dir / 'metrics.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-stage4', '--config', 'configs/experiment/stage4_tail_path.yaml',
        '--base-run', str(stage3_dir), '--run-id', STAGE4_RUN_ID,
        '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir),
    ], cwd=repo_dir, check=True)

FINAL_RUN_ID = 'stage6_mha_overlay_full_v001'
final_dir = artifact_dir / FINAL_RUN_ID
if not (final_dir / 'metrics.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-mha', '--config', 'configs/experiment/stage6_mha_overlay.yaml',
        '--base-run', str(stage4_dir), '--diagnostic-run', str(base_dir),
        '--run-id', FINAL_RUN_ID, '--artifact-dir', str(artifact_dir),
    ], cwd=repo_dir, check=True)
final_metrics = json.loads((final_dir / 'metrics.json').read_text())
final_metrics

In [ ]:
reference_dir = stage4_dir
if (reference_dir / 'metrics.json').is_file():
    reference = json.loads((reference_dir / 'metrics.json').read_text())
    comparison = {
        'stage4_reference_rmse': reference['pooled_rmse'],
        'stage6_candidate_rmse': final_metrics['pooled_rmse'],
        'rmse_delta': final_metrics['pooled_rmse'] - reference['pooled_rmse'],
        'reference_well_p90': reference['well_rmse_p90'],
        'candidate_well_p90': final_metrics['well_rmse_p90'],
        'reference_worst_10pct_sse_share': reference['worst_10pct_sse_share'],
        'candidate_worst_10pct_sse_share': final_metrics['worst_10pct_sse_share'],
    }
else:
    comparison = {'stage6_candidate_rmse': final_metrics['pooled_rmse'], 'note': 'Stage 4 reference artifact not found'}
comparison

## Promotion rule
The local reference moved from 12.204505 to 12.171786 (delta -0.032720), but the well-bootstrap 95% interval crosses zero. Treat this as a small candidate, not a guaranteed leaderboard gain. Send the native Colab comparison dictionary back before we port the overlay into the Kaggle execution Notebook.